# KdV — Learned Potential Evolution

Plot the evolution of the learned potential $V_\theta(u(t))$ over time
for 10 test trajectories.

- **Learned potential** (`s_onsagernet` model): $V_\theta(u) = V_0 + V_1 + V_2$

Each colored line is one test trajectory. A monotonically decreasing $V_\theta$
would indicate the model has learned a valid Lyapunov function for the dynamics.

In [ ]:
from pathlib import Path

import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.potential_evolution.helpers import (  # noqa: E402
    compute_V_learned,
    load_model_for_inference,
    load_test_data,
    plot_V_change,
    plot_V_evolution,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
RUNS_BASE = ROOT / "logs/official/runs/kdv"

print("Loading 'kdv' (real potential) ...")
kdv_model = load_model_for_inference(RUNS_BASE / "kdv", root=ROOT, device=device)

print("Loading 's_onsagernet' (learned potential) ...")
son_model = load_model_for_inference(RUNS_BASE / "s_onsagernet", root=ROOT, device=device)

kdv_potential = kdv_model.dynamics.potential  # KdVPotential
son_potential = son_model.dynamics.potential  # CoerciveAutogradPotentialV6
print(f"\nReal potential type   : {type(kdv_potential).__name__}")
print(f"Learned potential type: {type(son_potential).__name__}")

In [ ]:
test_data, t_coord, x_coord = load_test_data(str(ROOT / "data/kdv/*.hdf5"))
N_test, T, n_vars, Nx = test_data.shape
print(f"Test set: {N_test} trajectories  |  T={T}, n_vars={n_vars}, Nx={Nx}")

In [ ]:
import numpy as np

V_real = np.zeros((N_test, T), dtype=np.float64)
with torch.no_grad():
    for i, u_traj in enumerate(test_data):
        u_traj = u_traj.to(device)
        V_real[i] = kdv_potential.V(u_traj.squeeze(1)).squeeze(-1).cpu().numpy()

V_learned = compute_V_learned(son_potential, test_data, device)

print(f"V_real    shape: {V_real.shape}")
print(f"V_learned shape: {V_learned.shape}")

In [ ]:
plot_V_evolution(V_learned, ROOT / "figs/potential_evolution/kdv_potential_evolution.pdf")

In [ ]:
plot_V_change(V_learned, ROOT / "figs/potential_evolution/kdv_potential_change.pdf")